# clock

> when a repeating thing next comes due, and what to do about the fires you missed

In [ ]:
#| default_exp clock

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

The arithmetic only. Nothing here reads or writes a row. The schedule table can hold whatever
columns it likes, as long as it hands over a dict with `next_run` and either `every` or `cron`.

The part that matters is the catch-up policy. A laptop shut for nine days comes back with nine
missed 5pm fires, and running all nine is almost never what was wanted. `once` runs the latest
and forgets the rest, which is the right default for a roundup. `all` runs every one, for a
schedule where each fire does different work. `skip` runs none and just moves the clock on.

In [ ]:
#| export
import re, time
from datetime import datetime
from zoneinfo import ZoneInfo
from croniter import croniter
from fastcore.all import ifnone

In [ ]:
#| export
LATE_AFTER = 90.     # seconds past a fire before it is called late: the machine was off, not busy
MAX_FIRES = 1000     # fires walked for one catch-up. Past this the gap is jumped, not counted
CATCHUP = ('once', 'all', 'skip')

_DUR = re.compile(r'([\d.]+)\s*([smhdw])', re.I)
_MULT = dict(s=1, m=60, h=3600, d=86400, w=604800)

## Durations

In [ ]:
#| export
def secs(every) -> float:
    "Seconds from `'30m'`, `'6h'`, `'2 days'`, `'1w'`, `'1h30m'`, or a number of seconds."
    try: return float(every)
    except (TypeError, ValueError): pass
    if not (ms := _DUR.findall(str(every))): raise ValueError(f'not a duration: {every!r}')
    return sum(float(n) * _MULT[u.lower()] for n, u in ms)

In [ ]:
test_eq(secs(90), 90)
test_eq(secs('30m'), 1800)
test_eq(secs('6h'), 21600)
test_eq(secs('1h30m'), 5400)
test_eq(secs('2 days'), 172800)
test_fail(lambda: secs('soon'))

## Wall clock

`valid_cron` and `valid_tz` are for registration time. A schedule with a typo in it should be
refused when you write it, not at 5pm when nobody is watching.

In [ ]:
#| export
def valid_cron(cron:str) -> bool:
    'Whether `cron` is an expression this can read. Check when it is registered, not at 5pm.'
    return croniter.is_valid(str(cron))

def valid_tz(tz:str) -> bool:
    'Whether `tz` names a zone this can read.'
    try: ZoneInfo(str(tz)); return True
    except Exception: return False

def next_fire(cron:str,          # five-field cron expression, e.g. `'0 17 * * 1-5'`
              tz:str=None,       # IANA zone the fields are read in; the machine's when unset
              after:float=None   # epoch seconds to start from; now when unset
) -> float:
    'Epoch seconds of the first `cron` fire strictly after `after`.'
    z = ZoneInfo(tz) if tz else None
    t = datetime.fromtimestamp(ifnone(after, time.time()), z)
    return croniter(cron, t).get_next(datetime).timestamp()

def late(fire_at:float,            # when the fire was due
         at:float,                 # when it is being run
         after:float=LATE_AFTER    # seconds of lateness that still count as on time
) -> bool:
    'Whether this fire is running late enough to say so.'
    return (at - fire_at) > after

In [ ]:
test_eq(valid_cron('0 17 * * 1-5'), True)
test_eq(valid_cron('every friday'), False)
test_eq(valid_tz('Asia/Kolkata'), True)
test_eq(valid_tz('Mars/Olympus'), False)

5pm on a weekday in Kolkata, read from a Tuesday morning:

In [ ]:
tue = datetime(2026, 8, 25, 9, 0, tzinfo=ZoneInfo('Asia/Kolkata')).timestamp()
got = datetime.fromtimestamp(next_fire('0 17 * * 1-5', 'Asia/Kolkata', tue), ZoneInfo('Asia/Kolkata'))
test_eq((got.hour, got.minute, got.day), (17, 0, 25))

The zone is read at every fire, not once. A schedule keeps its wall-clock hour across a
daylight-saving change rather than sliding by an hour:

In [ ]:
ny = ZoneInfo('America/New_York')
before = datetime(2026, 3, 6, 12, 0, tzinfo=ny).timestamp()    # a Friday, before the change
after  = datetime(2026, 3, 10, 12, 0, tzinfo=ny).timestamp()   # a Tuesday, after it
for t in (before, after):
    got = datetime.fromtimestamp(next_fire('0 17 * * *', 'America/New_York', t), ny)
    test_eq((got.hour, got.minute), (17, 0))

## What came due

In [ ]:
#| export
def due_fires(sched:dict,          # `next_run`, and either `every` seconds or a `cron` with optional `tz`
              at:float,            # when the schedule is being looked at
              catchup:str='once'   # `once` runs the latest missed fire, `all` runs every one, `skip` none
) -> tuple:
    'The fire times to run now, and what `next_run` becomes.'
    nxt = sched['next_run']
    if nxt > at: return [], nxt
    got = []
    if sched.get('cron'):
        cron, tz = sched['cron'], sched.get('tz') or None
        while nxt <= at and len(got) < MAX_FIRES:
            got.append(nxt)
            nxt = next_fire(cron, tz, nxt)
        # a by-the-minute schedule left due for a year is jumped, not walked
        if nxt <= at: nxt = next_fire(cron, tz, at)
    else:
        every = max(secs(sched['every']), 1.)
        n = min(int((at - nxt)//every) + 1, MAX_FIRES)
        got = [nxt + i*every for i in range(n)]
        nxt = got[-1] + every
        if nxt <= at: nxt = at + every
    if catchup == 'skip': return [], nxt
    if catchup == 'once': return got[-1:], nxt
    return got, nxt

Nothing due yet is the common answer, and it costs no arithmetic:

In [ ]:
test_eq(due_fires(dict(next_run=200, every=10), at=100), ([], 200))

Four fires missed. `once` runs the last, `all` runs the lot, `skip` runs none, and all three
leave `next_run` in the same place:

In [ ]:
s = dict(next_run=100., every=10)
test_eq(due_fires(s, at=135, catchup='once'), ([130.], 140.))
test_eq(due_fires(s, at=135, catchup='all'),  ([100., 110., 120., 130.], 140.))
test_eq(due_fires(s, at=135, catchup='skip'), ([], 140.))

The same three answers on a wall clock. Nine days of a shut laptop is nine missed 5pm fires:

In [ ]:
kol = ZoneInfo('Asia/Kolkata')
first_run = datetime(2026, 8, 15, 17, 0, tzinfo=kol).timestamp()
back = datetime(2026, 8, 24, 9, 0, tzinfo=kol).timestamp()
s = dict(next_run=first_run, cron='0 17 * * *', tz='Asia/Kolkata')

fires_all, nxt = due_fires(s, at=back, catchup='all')
test_eq(len(fires_all), 9)
test_eq(due_fires(s, at=back, catchup='once')[0], fires_all[-1:])
test_eq(datetime.fromtimestamp(nxt, kol).day, 24)     # today's 5pm, still to come

A by-the-minute schedule left for a year does not walk half a million fires:

In [ ]:
yr = dict(next_run=0., cron='* * * * *')
got, nxt = due_fires(yr, at=365*86400, catchup='all')
test_eq(len(got), MAX_FIRES)
test(nxt, 365*86400, lambda a,b: a > b)     # jumped past the gap rather than counted through it

In [ ]:
test_eq(late(100, 150), False)
test_eq(late(100, 1000), True)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()